# PFA - Prédiction Nutritionnelle v4.1
**Auteur** : Aymane ABOUNAY - EMSI 4IIR G6 AI/DS

### Changements v4.1 (par rapport à v4) :
- **Huber Loss** au lieu de MSE → plus robuste aux outliers (plats >1000 kcal)
- **Gradient Clipping** (max_norm=1.0) → élimine les pics de train loss en Phase 2
- **Mass comme 5ème target** → multi-task learning, mass corrélée aux calories
- **LR Warmup** (3 epochs) → montée progressive du lr au début de chaque phase

### Conservé de v4 :
- Z-score normalization des targets
- Head simplifié (Dropout → Dense)
- EfficientNet-B3
- Dégel sélectif en Phase 2

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR, SequentialLR
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Configuration

In [ ]:
class Config:
    BASE_DIR = '/kaggle/input/datasets/gillesokhin/nutrition5k-dataset'
    CSV_PATH = f'{BASE_DIR}/dish_nutrition_values.csv'
    IMG_OVERHEAD = f'{BASE_DIR}/imagery/realsense_overhead'
    IMG_SIDE     = f'{BASE_DIR}/imagery/side_angles'
    OUTPUT_DIR   = '/kaggle/working'
    
    # 5 targets maintenant (mass ajouté)
    TARGET_COLS = ['calories', 'mass', 'protein', 'fat', 'carb']
    NUM_TARGETS = 5
    # Indices pour les 4 targets nutritionnels (sans mass) pour le reporting
    NUTRITION_COLS = ['calories', 'protein', 'fat', 'carb']
    NUTRITION_IDX  = [0, 2, 3, 4]  # indices dans TARGET_COLS
    
    # EfficientNet-B3
    IMG_SIZE   = 300
    BATCH_SIZE = 24
    
    # Phase 1: Head-only
    P1_EPOCHS = 20
    P1_LR     = 1e-3
    
    # Phase 2: Selective fine-tuning
    P2_EPOCHS = 20
    P2_LR     = 1e-4
    
    # Early stopping
    PATIENCE = 7
    
    # Gradient clipping
    MAX_GRAD_NORM = 1.0
    
    # LR Warmup
    WARMUP_EPOCHS = 3
    
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

cfg = Config()
print('Config v4.1 OK')
print(f'  Targets: {cfg.TARGET_COLS} ({cfg.NUM_TARGETS} targets)')
print(f'  Gradient clipping: max_norm={cfg.MAX_GRAD_NORM}')
print(f'  LR Warmup: {cfg.WARMUP_EPOCHS} epochs')

## 3. Construction du Dataset

In [ ]:
df_all = pd.read_csv(cfg.CSV_PATH)
print(f'CSV total: {len(df_all)} plats')
print(f'Colonnes: {list(df_all.columns)}')

image_records = []

# 1) Overhead images (prioritaires)
overhead_dishes = set()
for _, row in tqdm(df_all.iterrows(), total=len(df_all), desc='Overhead'):
    dish_id = row['dish_id']
    img_path = os.path.join(cfg.IMG_OVERHEAD, str(dish_id), 'rgb.png')
    if os.path.exists(img_path):
        image_records.append({
            'dish_id': dish_id,
            'img_path': img_path,
            'source': 'overhead'
        })
        overhead_dishes.add(dish_id)

print(f'Overhead: {len(overhead_dishes)} plats')

# 2) Side angles UNIQUEMENT pour les plats SANS overhead
side_only_count = 0
for _, row in tqdm(df_all.iterrows(), total=len(df_all), desc='Side (fallback)'):
    dish_id = row['dish_id']
    if dish_id in overhead_dishes:
        continue
    
    side_dir = os.path.join(cfg.IMG_SIDE, str(dish_id))
    if not os.path.exists(side_dir):
        continue
    
    files = sorted([f for f in os.listdir(side_dir) 
                    if f.startswith('camera_A') and f.endswith('.jpeg')])
    if files:
        mid_frame = files[len(files) // 2]
        image_records.append({
            'dish_id': dish_id,
            'img_path': os.path.join(side_dir, mid_frame),
            'source': 'side_fallback'
        })
        side_only_count += 1

print(f'Side (fallback): {side_only_count}')
print(f'TOTAL: {len(image_records)} images (1 par plat)')

df_images = pd.DataFrame(image_records)
df_images = df_images.merge(df_all[['dish_id'] + cfg.TARGET_COLS], on='dish_id', how='inner')
print(f'DataFrame final: {len(df_images)} lignes')

# V\u00e9rifier la corr\u00e9lation mass/calories
corr = df_images['mass'].corr(df_images['calories'])
print(f'\nCorr\u00e9lation mass/calories: {corr:.3f} (multi-task devrait aider!)')

## 4. Split Train / Val / Test (70/15/15)

In [ ]:
train_df, temp_df = train_test_split(df_images, test_size=0.30, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 5. Z-Score Normalization des Targets (5 targets)

In [ ]:
# Calculer mean/std sur le train set (5 targets maintenant)
target_mean = train_df[cfg.TARGET_COLS].mean().values.astype(np.float32)
target_std  = train_df[cfg.TARGET_COLS].std().values.astype(np.float32)

print('Target normalization stats (from train set):')
for i, col in enumerate(cfg.TARGET_COLS):
    print(f'  {col:<12} mean={target_mean[i]:>8.2f}  std={target_std[i]:>8.2f}')

target_mean_t = torch.tensor(target_mean)
target_std_t  = torch.tensor(target_std)

# V\u00e9rifier
train_normalized = (train_df[cfg.TARGET_COLS].values - target_mean) / target_std
print(f'\nApr\u00e8s normalisation (train):')
print(f'  Mean: {train_normalized.mean(axis=0).round(3)}')
print(f'  Std:  {train_normalized.std(axis=0).round(3)}')

In [ ]:
# Visualiser avant/apr\u00e8s normalisation (5 targets)
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for i, col in enumerate(cfg.TARGET_COLS):
    axes[0][i].hist(train_df[col], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0][i].set_title(f'{col} (brut)\n\u03bc={target_mean[i]:.1f}, \u03c3={target_std[i]:.1f}', fontsize=9)
    axes[1][i].hist(train_normalized[:, i], bins=50, edgecolor='black', alpha=0.7, color='coral')
    axes[1][i].set_title(f'{col} (z-score)', fontsize=9)
axes[0][0].set_ylabel('Brut', fontsize=12)
axes[1][0].set_ylabel('Z-Score', fontsize=12)
plt.suptitle('Distribution des 5 targets avant/apr\u00e8s Z-Score (Train)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/zscore_dist_v4_1.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Dataset & DataLoaders

In [ ]:
class Nutrition5KDataset(Dataset):
    def __init__(self, dataframe, target_cols, transform=None, target_mean=None, target_std=None):
        self.df = dataframe.reset_index(drop=True)
        self.target_cols = target_cols
        self.transform = transform
        self.target_mean = target_mean
        self.target_std = target_std
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['img_path']).convert('RGB')
        
        raw_label = np.array([row[c] for c in self.target_cols], dtype=np.float32)
        
        if self.target_mean is not None and self.target_std is not None:
            label = (raw_label - self.target_mean) / self.target_std
        else:
            label = raw_label
        
        label = torch.tensor(label, dtype=torch.float32)
        
        if self.transform:
            img = self.transform(img)
        return img, label


def get_transforms(img_size=300, is_train=True):
    resize_size = max(img_size + 32, 256)
    if is_train:
        return transforms.Compose([
            transforms.Resize(resize_size),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(cfg.IMAGENET_MEAN, cfg.IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(resize_size),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(cfg.IMAGENET_MEAN, cfg.IMAGENET_STD),
        ])


train_ds = Nutrition5KDataset(train_df, cfg.TARGET_COLS, get_transforms(cfg.IMG_SIZE, True),  target_mean, target_std)
val_ds   = Nutrition5KDataset(val_df,   cfg.TARGET_COLS, get_transforms(cfg.IMG_SIZE, False), target_mean, target_std)
test_ds  = Nutrition5KDataset(test_df,  cfg.TARGET_COLS, get_transforms(cfg.IMG_SIZE, False), target_mean, target_std)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

imgs, labels = next(iter(train_loader))
print(f'Batch images: {imgs.shape}')
print(f'Batch labels (5 targets, normalized): {labels.shape}')
print(f'Targets order: {cfg.TARGET_COLS}')

## 7. Modèle v4.1

In [ ]:
class NutritionRegressorV41(nn.Module):
    """
    EfficientNet-B3 + head simplifi\u00e9.
    Head: GAP (built-in) \u2192 Dropout(0.3) \u2192 Dense(5)
    
    5 targets: calories, mass, protein, fat, carb
    Mass aide le mod\u00e8le via multi-task learning.
    """
    def __init__(self, num_targets=5, dropout=0.3):
        super().__init__()
        base = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        in_features = base.classifier[1].in_features  # 1536
        base.classifier = nn.Identity()
        self.backbone = base
        
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_targets)
        )
    
    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Backbone gel\u00e9. Trainable params: {trainable:,}')
    
    def unfreeze_backbone_selective(self, num_layers_to_unfreeze=100):
        all_layers = list(self.backbone.parameters())
        for p in all_layers:
            p.requires_grad = False
        for p in all_layers[-num_layers_to_unfreeze:]:
            p.requires_grad = True
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Backbone: {num_layers_to_unfreeze} derni\u00e8res couches d\u00e9gel\u00e9es. Trainable params: {trainable:,}')
    
    def forward(self, x):
        return self.head(self.backbone(x))


model = NutritionRegressorV41(num_targets=cfg.NUM_TARGETS, dropout=0.3).to(device)

# Huber Loss (delta=1.0) — robuste aux outliers
criterion = nn.HuberLoss(delta=1.0)

total_params = sum(p.numel() for p in model.parameters())
head_params = sum(p.numel() for p in model.head.parameters())
print(f'Total params: {total_params/1e6:.1f}M')
print(f'Head params:  {head_params:,}')
print(f'Loss: Huber (delta=1.0) — robuste aux outliers')
print(f'Targets: {cfg.NUM_TARGETS} ({cfg.TARGET_COLS})')

## 8. Fonctions d'Entraînement (avec Gradient Clipping)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.should_stop = False
    
    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False


def get_scheduler_with_warmup(optimizer, warmup_epochs, total_epochs):
    """Cr\u00e9e un scheduler avec warmup lin\u00e9aire puis cosine annealing."""
    warmup_scheduler = LambdaLR(
        optimizer, 
        lr_lambda=lambda epoch: (epoch + 1) / warmup_epochs
    )
    cosine_scheduler = CosineAnnealingLR(
        optimizer, 
        T_max=total_epochs - warmup_epochs
    )
    scheduler = SequentialLR(
        optimizer, 
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs]
    )
    return scheduler


def train_one_epoch(model, loader, criterion, optimizer, device, max_grad_norm=1.0):
    model.train()
    total_loss = 0
    all_preds, all_targets = [], []
    
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        
        optimizer.step()
        
        total_loss += loss.item() * imgs.size(0)
        all_preds.append(preds.detach().cpu())
        all_targets.append(labels.detach().cpu())
    
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    mae_real = (torch.abs(all_preds - all_targets) * target_std_t).mean(dim=0)
    return total_loss / len(loader.dataset), mae_real


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    
    for imgs, labels in tqdm(loader, desc='Eval', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        preds = model(imgs)
        loss = criterion(preds, labels)
        
        total_loss += loss.item() * imgs.size(0)
        all_preds.append(preds.cpu())
        all_targets.append(labels.cpu())
    
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    
    # D\u00e9normaliser
    preds_real = all_preds * target_std_t + target_mean_t
    targets_real = all_targets * target_std_t + target_mean_t
    mae_real  = torch.abs(preds_real - targets_real).mean(dim=0)
    rmse_real = torch.sqrt(((preds_real - targets_real) ** 2).mean(dim=0))
    
    return total_loss / len(loader.dataset), mae_real, rmse_real, preds_real, targets_real


print('Fonctions v4.1 OK (Huber + Gradient Clipping + Warmup)')

## 9. Phase 1 : Head-only (20 epochs max, warmup 3 epochs)

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_mae_real': [], 'phase': [], 'lr': []}

# Indices pour afficher les 4 nutriments (sans mass)
ni = cfg.NUTRITION_IDX  # [0, 2, 3, 4] = calories, protein, fat, carb

print('='*70)
print('PHASE 1: Head-only (20 epochs max, patience=7, warmup=3)')
print('='*70)
model.freeze_backbone()

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg.P1_LR, weight_decay=1e-4
)
scheduler = get_scheduler_with_warmup(optimizer, cfg.WARMUP_EPOCHS, cfg.P1_EPOCHS)
early_stop = EarlyStopping(patience=cfg.PATIENCE)
best_val_loss = float('inf')

for epoch in range(cfg.P1_EPOCHS):
    current_lr = optimizer.param_groups[0]['lr']
    train_loss, train_mae = train_one_epoch(model, train_loader, criterion, optimizer, device, cfg.MAX_GRAD_NORM)
    val_loss, val_mae, val_rmse, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mae_real'].append(val_mae.numpy().tolist())
    history['phase'].append(1)
    history['lr'].append(current_lr)
    
    improved = early_stop.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'{cfg.OUTPUT_DIR}/best_model_v4_1.pth')
    
    marker = ' \u2605' if improved else ''
    print(f'  Epoch {epoch+1:02d}/{cfg.P1_EPOCHS} | lr={current_lr:.1e} | '
          f'TrL: {train_loss:.4f} | VaL: {val_loss:.4f} | '
          f'Cal:{val_mae[ni[0]]:.1f} P:{val_mae[ni[1]]:.1f} F:{val_mae[ni[2]]:.1f} C:{val_mae[ni[3]]:.1f} '
          f'Mass:{val_mae[1]:.1f}'
          f'{marker} (es:{early_stop.counter}/{cfg.PATIENCE})')
    
    if early_stop.should_stop:
        print(f'\n  \u26d4 Early stopping \u00e0 epoch {epoch+1}')
        break

p1_epochs_done = epoch + 1
print(f'\nPhase 1: {p1_epochs_done} epochs, best val loss: {best_val_loss:.4f}')

## 10. Phase 2 : Selective Fine-tuning (20 epochs max)

In [ ]:
model.load_state_dict(torch.load(f'{cfg.OUTPUT_DIR}/best_model_v4_1.pth'))

print('='*70)
print('PHASE 2: Selective Fine-tuning (100 couches, warmup=3, grad_clip=1.0)')
print('='*70)

model.unfreeze_backbone_selective(num_layers_to_unfreeze=100)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg.P2_LR, weight_decay=1e-4
)
scheduler = get_scheduler_with_warmup(optimizer, cfg.WARMUP_EPOCHS, cfg.P2_EPOCHS)
early_stop2 = EarlyStopping(patience=cfg.PATIENCE)

for epoch in range(cfg.P2_EPOCHS):
    current_lr = optimizer.param_groups[0]['lr']
    train_loss, train_mae = train_one_epoch(model, train_loader, criterion, optimizer, device, cfg.MAX_GRAD_NORM)
    val_loss, val_mae, val_rmse, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mae_real'].append(val_mae.numpy().tolist())
    history['phase'].append(2)
    history['lr'].append(current_lr)
    
    improved = early_stop2.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'{cfg.OUTPUT_DIR}/best_model_v4_1.pth')
    
    marker = ' \u2605' if improved else ''
    print(f'  Epoch {epoch+1:02d}/{cfg.P2_EPOCHS} | lr={current_lr:.1e} | '
          f'TrL: {train_loss:.4f} | VaL: {val_loss:.4f} | '
          f'Cal:{val_mae[ni[0]]:.1f} P:{val_mae[ni[1]]:.1f} F:{val_mae[ni[2]]:.1f} C:{val_mae[ni[3]]:.1f} '
          f'Mass:{val_mae[1]:.1f}'
          f'{marker} (es:{early_stop2.counter}/{cfg.PATIENCE})')
    
    if early_stop2.should_stop:
        print(f'\n  \u26d4 Early stopping \u00e0 epoch {epoch+1}')
        break

print(f'\nMeilleur Val Loss global: {best_val_loss:.4f}')

## 11. Évaluation Finale (Test Set)

In [ ]:
model.load_state_dict(torch.load(f'{cfg.OUTPUT_DIR}/best_model_v4_1.pth'))
test_loss, test_mae, test_rmse, test_preds, test_targets = evaluate(
    model, test_loader, criterion, device
)

print('='*60)
print('R\u00c9SULTATS FINAUX v4.1 - Huber + GradClip + Mass (Test Set)')
print('='*60)
print(f'{"Target":<12} {"MAE":>8} {"RMSE":>8}')
print('-'*30)
for i, col in enumerate(cfg.TARGET_COLS):
    print(f'{col:<12} {test_mae[i]:>8.2f} {test_rmse[i]:>8.2f}')
print('-'*30)

# Comparaison sur les 4 nutriments principaux (sans mass)
v1_mae  = [82.09, 9.40, 5.95, 8.02]
v2_mae  = [108.25, 8.67, 7.41, 10.19]
v3_mae  = [84.86, 8.54, 6.63, 9.67]
v4_mae  = [80.36, 5.58, 6.42, 7.55]
v41_mae = [test_mae[ni[0]].item(), test_mae[ni[1]].item(), test_mae[ni[2]].item(), test_mae[ni[3]].item()]

print(f'\n--- Comparaison v1 / v2 / v3 / v4 / v4.1 ---')
print(f'{"Nutriment":<12} {"v1":>8} {"v2":>8} {"v3":>8} {"v4":>8} {"v4.1":>8}')
print('-'*60)
for i, col in enumerate(cfg.NUTRITION_COLS):
    all_mae = [v1_mae[i], v2_mae[i], v3_mae[i], v4_mae[i], v41_mae[i]]
    best = min(all_mae)
    v41_str = f'{v41_mae[i]:.2f}'
    if v41_mae[i] <= best + 0.01:
        v41_str += ' \u2605'
    print(f'{col:<12} {v1_mae[i]:>8.2f} {v2_mae[i]:>8.2f} {v3_mae[i]:>8.2f} {v4_mae[i]:>8.2f} {v41_str:>8}')

print(f'\nMass MAE: {test_mae[1]:.2f} g (target auxiliaire)')

## 12. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
epochs_range = range(1, len(history['train_loss'])+1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], label='Val Loss', linewidth=2)
if 2 in history['phase']:
    p1_end = history['phase'].index(2)
    axes[0].axvline(x=p1_end + 0.5, color='red', linestyle='--', alpha=0.5, label='Phase 1\u21922')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Huber Loss (normalized)')
axes[0].set_title('v4.1 - Courbe de Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE r\u00e9elle (4 nutriments)
mae_hist = np.array(history['val_mae_real'])
for idx, col in zip(cfg.NUTRITION_IDX, cfg.NUTRITION_COLS):
    axes[1].plot(epochs_range, mae_hist[:, idx], label=col, linewidth=2)
if 2 in history['phase']:
    axes[1].axvline(x=p1_end + 0.5, color='red', linestyle='--', alpha=0.5, label='Phase 1\u21922')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (valeurs r\u00e9elles)')
axes[1].set_title('v4.1 - MAE par nutriment')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(epochs_range, history['lr'], linewidth=2, color='green')
if 2 in history['phase']:
    axes[2].axvline(x=p1_end + 0.5, color='red', linestyle='--', alpha=0.5, label='Phase 1\u21922')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('v4.1 - Learning Rate (warmup + cosine)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/training_curves_v4_1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: Pr\u00e9dit vs R\u00e9el (5 targets)
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for i, col in enumerate(cfg.TARGET_COLS):
    ax = axes[i]
    real = test_targets[:, i].numpy()
    pred = test_preds[:, i].numpy()
    ax.scatter(real, pred, alpha=0.3, s=8)
    lims = [min(real.min(), pred.min()), max(real.max(), pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=2, label='y=x')
    ax.set_xlabel(f'{col} (R\u00e9el)')
    ax.set_ylabel(f'{col} (Pr\u00e9dit)')
    ax.set_title(f'{col}\nMAE={test_mae[i]:.2f}')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle('v4.1 - Pr\u00e9dit vs R\u00e9el (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/scatter_v4_1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Exemples de pr\u00e9dictions (affichage 4 nutriments principaux)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
indices = np.random.choice(len(test_ds), 10, replace=False)
model.eval()

for idx_plot, idx_data in enumerate(indices):
    ax = axes[idx_plot // 5][idx_plot % 5]
    img, label_norm = test_ds[idx_data]
    
    with torch.no_grad():
        pred_norm = model(img.unsqueeze(0).to(device)).cpu().squeeze()
    
    label = label_norm * target_std_t + target_mean_t
    pred  = pred_norm * target_std_t + target_mean_t
    
    img_show = img.clone()
    for c in range(3):
        img_show[c] = img_show[c] * cfg.IMAGENET_STD[c] + cfg.IMAGENET_MEAN[c]
    img_show = img_show.permute(1, 2, 0).clamp(0, 1)
    
    ax.imshow(img_show)
    # Afficher cal/prot/fat/carb (indices 0,2,3,4)
    ax.set_title(
        f'R\u00e9el: {label[0]:.0f}/{label[2]:.0f}/{label[3]:.0f}/{label[4]:.0f}\n'
        f'Pr\u00e9d: {pred[0]:.0f}/{pred[2]:.0f}/{pred[3]:.0f}/{pred[4]:.0f}',
        fontsize=8
    )
    ax.axis('off')

plt.suptitle('Cal/Prot/Fat/Carb \u2014 R\u00e9el vs Pr\u00e9dit (v4.1)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/prediction_examples_v4_1.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Sauvegarde du Modèle Final

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'backbone': 'efficientnet_b3',
    'head': 'simple (Dropout + Dense)',
    'img_size': cfg.IMG_SIZE,
    'target_cols': cfg.TARGET_COLS,
    'nutrition_cols': cfg.NUTRITION_COLS,
    'nutrition_idx': cfg.NUTRITION_IDX,
    'imagenet_mean': cfg.IMAGENET_MEAN,
    'imagenet_std': cfg.IMAGENET_STD,
    'target_mean': target_mean.tolist(),
    'target_std': target_std.tolist(),
    'test_mae': test_mae.numpy().tolist(),
    'test_rmse': test_rmse.numpy().tolist(),
    'num_images': len(df_images),
    'num_train': len(train_df),
    'loss': 'HuberLoss(delta=1.0)',
    'grad_clip': cfg.MAX_GRAD_NORM,
    'version': 'v4.1',
}
torch.save(checkpoint, f'{cfg.OUTPUT_DIR}/nutrition_model_v4_1_final.pth')
print(f'Mod\u00e8le sauvegard\u00e9: nutrition_model_v4_1_final.pth')
print(f'Taille: {os.path.getsize(f"{cfg.OUTPUT_DIR}/nutrition_model_v4_1_final.pth") / 1e6:.1f} MB')

## 14. Fonction d'Inférence (4 nutriments, ignore mass)

In [ ]:
def predict_nutrition(image_path, model, device, target_mean, target_std, img_size=300):
    """Pr\u00e9dit les valeurs nutritionnelles d'une image.
    
    Le mod\u00e8le pr\u00e9dit 5 targets (cal, mass, prot, fat, carb).
    On retourne uniquement les 4 nutriments (sans mass).
    """
    transform = transforms.Compose([
        transforms.Resize(img_size + 32),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        pred_norm = model(img_tensor).cpu().squeeze().numpy()
    
    pred_real = pred_norm * target_std + target_mean
    
    # Indices: 0=calories, 1=mass, 2=protein, 3=fat, 4=carb
    return {
        'calories': max(0, float(pred_real[0])),
        'protein':  max(0, float(pred_real[2])),
        'fat':      max(0, float(pred_real[3])),
        'carb':     max(0, float(pred_real[4])),
    }

# Test
sample = test_df.iloc[0]
result = predict_nutrition(sample['img_path'], model, device, target_mean, target_std)
print(f'Dish: {sample["dish_id"]}')
print(f'R\u00e9el  \u2192 Cal:{sample["calories"]:.1f} P:{sample["protein"]:.1f} F:{sample["fat"]:.1f} C:{sample["carb"]:.1f}')
print(f'Pr\u00e9dit \u2192 Cal:{result["calories"]:.1f} P:{result["protein"]:.1f} F:{result["fat"]:.1f} C:{result["carb"]:.1f}')

In [ ]:
print('\n' + '='*60)
print('R\u00c9SUM\u00c9 v4.1')
print('='*60)
print(f'Dataset: {len(df_images)} images')
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Targets: {cfg.TARGET_COLS} (5 targets, mass auxiliaire)')
print(f'Normalization: Z-score (mean/std du train)')
print(f'Head: Dropout(0.3) \u2192 Dense(5) ({head_params:,} params)')
print(f'Loss: Huber (delta=1.0)')
print(f'Gradient clipping: max_norm={cfg.MAX_GRAD_NORM}')
print(f'LR Warmup: {cfg.WARMUP_EPOCHS} epochs')
print(f'Phase 1: {p1_epochs_done} epochs (lr={cfg.P1_LR})')
print(f'Phase 2: selective fine-tuning (lr={cfg.P2_LR})')
print(f'\nMAE Calories:  {test_mae[0]:.2f} kcal')
print(f'MAE Mass:      {test_mae[1]:.2f} g')
print(f'MAE Prot\u00e9ines: {test_mae[2]:.2f} g')
print(f'MAE Lipides:   {test_mae[3]:.2f} g')
print(f'MAE Glucides:  {test_mae[4]:.2f} g')